# ANON AI ULTRA — Hazır sohbet modelini test et

Bu defter mevcut `ANON-AI-ULTRA-chat.zip` dosyasını açar ve beş Türkçe soruyla test eder. Eğitim yapmaz; yalnızca yükleme ve metin üretimini kontrol eder.

In [ ]:
import os, sys, subprocess, torch
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'safetensors'], check=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU yok; CPU kullanılacak')

## 1. Sohbet ZIP dosyasını yükle

Telefonundan indirdiğin `ANON-AI-ULTRA-chat.zip` dosyasını seç.

In [ ]:
from google.colab import files
import zipfile, shutil
uploaded = files.upload()
zip_candidates = [name for name in uploaded if name.lower().endswith('.zip')]
assert zip_candidates, 'ZIP dosyası seçilmedi.'
ZIP_PATH = os.path.abspath(zip_candidates[0])
MODEL_DIR = '/content/anon-ai-chat-test-model'
shutil.rmtree(MODEL_DIR, ignore_errors=True)
os.makedirs(MODEL_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall(MODEL_DIR)
from pathlib import Path
configs = list(Path(MODEL_DIR).rglob('config.json'))
assert configs, 'ZIP içinde config.json bulunamadı.'
MODEL_PATH = str(configs[0].parent)
print('Model klasörü:', MODEL_PATH)
print('Model dosyası bulundu:', (Path(MODEL_PATH) / 'model.safetensors').exists() or (Path(MODEL_PATH) / 'pytorch_model.bin').exists())

## 2. Modeli yükle ve beş soruyu sor

Bu testin amacı cevap üretimini görmek. Hücrenin çalışması, cevapların doğru veya kaliteli olduğunu tek başına kanıtlamaz.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()
model.eval()

print("Tokenizer vocab:", len(tokenizer))
print("Model embedding rows:", model.get_input_embeddings().num_embeddings)
assert len(tokenizer) == model.get_input_embeddings().num_embeddings, "Tokenizer/model vocab mismatch."
assert tokenizer.eos_token_id is not None and tokenizer.pad_token_id is not None, "EOS/PAD token is missing."
for marker in ["<|system|>", "<|user|>", "<|assistant|>"]:
    marker_ids = tokenizer.encode(marker, add_special_tokens=False)
    assert len(marker_ids) == 1 and marker_ids[0] != tokenizer.unk_token_id, f"Invalid chat marker: {marker}"
probe = "Türkçe karakter testi: ğüşiöç İı"
assert tokenizer.decode(tokenizer.encode(probe, add_special_tokens=False), skip_special_tokens=False) == probe, "Tokenizer Turkish round-trip failed."
print("Tokenizer, chat markers and model dimensions: OK")

questions = [
    'Merhaba, nasılsın?',
    'Türkiye\'nin başkenti neresidir?',
    'Yapay zekâ nedir?',
    'Almanca merhaba nasıl söylenir?',
    'Bana kendini kısaca tanıtır mısın?'
]

print('ANON AI sohbet testi | Cihaz:', device)
print('=' * 60)
for i, question in enumerate(questions, 1):
    prompt = '<|user|>' + question + tokenizer.eos_token + '<|assistant|>'
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.inference_mode():
        generated = model.generate(
            **inputs, max_new_tokens=80, do_sample=False,
            repetition_penalty=1.15, no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    answer_tokens = generated[0, inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()
    print(f'\nTEST {i}: {question}')
    print('ANON AI:', answer if answer else '[Boş yanıt]')
    print('-' * 60)
print('Test tamamlandı. Cevapları kalite açısından ayrıca değerlendireceğiz.')